# Handwritten-style Text Generation with a Character-level RNN

**Goal:** learn patterns from text with a character-level RNN, generate *new* text, and render it as
handwriting learned from a dataset of handwritten characters.

The pipeline has two learned parts that meet at the character level:

| Stage | Model | Trained on | Output |
|---|---|---|---|
| 1. Text generator | 2-layer **LSTM** character-level language model | a text corpus | new text, one character at a time |
| 2. Handwriting renderer | **Conditional VAE** (character label + style vector → glyph image) | the *English Handwritten Characters* dataset (62 classes: a-z, A-Z, 0-9) | a handwritten glyph for any character, in many styles |

**Why this differs from the original notebook.** The original model regressed a 500×500 image (a 250k-unit dense layer, ~256M parameters)
from a single character with MSE loss. With only 55 photos per class, MSE regression converges to the *average* of those 55 photos,
so every "A" comes out as the same blurry smudge, and the LSTM had nothing to do (the input sequence was one character long).
Here the RNN does real sequence modelling, and a latent *style* vector gives each glyph natural variation.

In [ ]:
import os, re, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw, ImageFont

import tensorflow as tf
from tensorflow.keras import layers, Model

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## Configuration

In [ ]:
QUICK_TEST = bool(os.environ.get("QUICK_TEST"))   # tiny run to check that everything executes

# --- data locations (Kaggle paths) ---
DATA_ROOT  = "/kaggle/input/english-handwritten-characters-dataset"
CSV_PATH   = f"{DATA_ROOT}/english.csv"
CORPUS_PATH = None            # optional: path to any .txt file to train the text RNN on

# --- character set: exactly the 62 classes the handwriting dataset knows, plus space ---
CHARS = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
ALPHABET = [" "] + CHARS     # text-RNN vocabulary (63 symbols)

# --- text RNN ---
SEQ_LEN, RNN_UNITS, EMB_DIM = 64, 256, 64
RNN_EPOCHS = 2 if QUICK_TEST else 25
RNN_BATCH  = 128

# --- glyph model ---
IMG, LATENT = 64, 16
VAE_EPOCHS = 2 if QUICK_TEST else 150
VAE_BATCH  = 64
KL_BETA    = 0.5               # <1 favours sharper glyphs, >1 favours smoother latent space

# Part 1 - Character-level RNN for text

### 1.1 Corpus
Any plain-text file works. The loader tries, in order: `CORPUS_PATH`, the Shakespeare file used in the TensorFlow tutorials
(needs internet), and finally a small built-in passage. The built-in passage is only enough to prove the code runs; the RNN will
largely memorise it, so use a real corpus for interesting output.

Punctuation is stripped and whitespace collapsed because the handwriting dataset has no glyphs for punctuation.

In [ ]:
FALLBACK_TEXT = """
Handwriting is a trace of motion. Every letter is a small decision made by a hand that has repeated the same
shape thousands of times and still never draws it twice in exactly the same way. A careful writer slows down and
the loops grow round and even, a hurried writer leans forward and the words run together like water in a gutter.
Machines learn to imitate this slowly. First they learn which letter tends to follow which, then which words tend to
follow which, and only after that do they learn to turn those choices into marks on a page. A model that reads
text one character at a time keeps a memory of what it has seen so far and uses that memory to guess what comes next.
When it guesses well, the text it writes sounds like the text it studied. When it guesses badly, the words melt
into noise. The pen moves, the page fills, and a new sentence appears that nobody wrote before.
"""

def load_corpus():
    if CORPUS_PATH and os.path.exists(CORPUS_PATH):
        return open(CORPUS_PATH, encoding="utf-8", errors="ignore").read(), CORPUS_PATH
    try:
        p = tf.keras.utils.get_file(
            "shakespeare.txt",
            "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")
        return open(p, encoding="utf-8").read(), "shakespeare.txt"
    except Exception as e:
        print("Could not download a corpus (", type(e).__name__, ") - using the tiny built-in passage.")
        return FALLBACK_TEXT * 20, "built-in passage"

def clean(text):
    text = re.sub(r"[^A-Za-z0-9 ]+", " ", text)
    return re.sub(r" +", " ", text).strip()

raw, corpus_name = load_corpus()
corpus = clean(raw)
if QUICK_TEST:
    corpus = corpus[:50_000]
print(f"corpus: {corpus_name} | {len(corpus):,} characters")
print(corpus[:200])

### 1.2 Encode the text and build training windows
The task is next-character prediction: for a window `c[t..t+63]` the target is the same window shifted by one.

In [ ]:
c2i = {c: i for i, c in enumerate(ALPHABET)}
i2c = {i: c for c, i in c2i.items()}
VOCAB = len(ALPHABET)

ids = np.array([c2i[c] for c in corpus if c in c2i], dtype=np.int32)

n_windows = min(100_000, max(5_000, len(ids) // 2))
starts = np.random.randint(0, len(ids) - SEQ_LEN - 1, size=n_windows)
X = np.stack([ids[s : s + SEQ_LEN]         for s in starts])
Y = np.stack([ids[s + 1 : s + SEQ_LEN + 1] for s in starts])
n_val = n_windows // 20
train_ds = tf.data.Dataset.from_tensor_slices((X[n_val:], Y[n_val:])).shuffle(10_000).batch(RNN_BATCH).prefetch(2)
val_ds   = tf.data.Dataset.from_tensor_slices((X[:n_val],  Y[:n_val])).batch(RNN_BATCH)
print("windows:", X.shape, "| vocab:", VOCAB)

### 1.3 The model
Embedding → 2 stacked LSTMs → softmax over the 63 characters, with a prediction at every time step.

In [ ]:
def build_char_rnn():
    inp = layers.Input((None,), dtype="int32")
    x = layers.Embedding(VOCAB, EMB_DIM)(inp)
    x = layers.LSTM(RNN_UNITS, return_sequences=True)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(RNN_UNITS, return_sequences=True)(x)
    x = layers.Dropout(0.2)(x)
    return Model(inp, layers.Dense(VOCAB)(x), name="char_rnn")        # logits

rnn = build_char_rnn()
rnn.compile(
    optimizer=tf.keras.optimizers.Adam(2e-3, clipnorm=1.0),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["sparse_categorical_accuracy"])
rnn.summary()

In [ ]:
hist = rnn.fit(train_ds, validation_data=val_ds, epochs=RNN_EPOCHS, verbose=2,
               callbacks=[tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, verbose=1)])

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val")
ax[0].set_title("Cross-entropy loss"); ax[0].legend()
ax[1].plot(hist.history["sparse_categorical_accuracy"]); ax[1].plot(hist.history["val_sparse_categorical_accuracy"])
ax[1].set_title("Next-char accuracy"); plt.tight_layout(); plt.show()

### 1.4 Generating text
Sample one character at a time and feed it back in. **Temperature** controls risk: low values give safe, repetitive text;
high values give more surprising (and more error-prone) text.

In [ ]:
def generate_text(seed="The ", n_chars=120, temperature=0.8):
    ctx = [c2i[c] for c in seed if c in c2i] or [c2i[" "]]
    out = list(seed)
    for _ in range(n_chars):
        logits = rnn(np.array([ctx[-SEQ_LEN:]]), training=False)[0, -1].numpy() / temperature
        p = np.exp(logits - logits.max()); p /= p.sum()
        nxt = np.random.choice(VOCAB, p=p)
        ctx.append(nxt); out.append(i2c[nxt])
    return "".join(out)

for T in (0.5, 0.8, 1.1):
    print(f"--- temperature {T} ---")
    print(generate_text("The ", 150, T), "\n")

# Part 2 - Learning handwriting from the character dataset

### 2.1 Load the images
Each photo is converted to grayscale, auto-contrasted, padded to a square (so thin letters are not stretched), resized to 64×64 and
**inverted so that ink = 1 and paper = 0**. If the Kaggle dataset is not found, a synthetic font-rendered dataset is used so the notebook
still runs end to end (its output is *not* handwriting - attach the dataset for that).

In [ ]:
def preprocess(img):
    img = ImageOps.autocontrast(img.convert("L"))
    img = ImageOps.pad(img, (IMG, IMG), color=255, method=Image.BILINEAR)
    return 1.0 - np.asarray(img, dtype=np.float32) / 255.0

def load_real_dataset():
    df = pd.read_csv(CSV_PATH)
    if QUICK_TEST:
        df = df.groupby("label").head(10)
    imgs = [preprocess(Image.open(f"{DATA_ROOT}/{p}")) for p in df["image"]]
    return np.stack(imgs)[..., None], df["label"].astype(str).values

def load_synthetic_dataset(per_class=40):
    fonts = sorted(glob.glob("/usr/share/fonts/**/*.ttf", recursive=True))[:12]
    rng = np.random.default_rng(0); imgs, labels = [], []
    for ch in CHARS:
        for k in range(per_class):
            canvas = Image.new("L", (96, 96), 255)
            font = ImageFont.truetype(fonts[k % len(fonts)], int(rng.integers(44, 60))) if fonts else ImageFont.load_default()
            ImageDraw.Draw(canvas).text((int(rng.integers(8, 28)), int(rng.integers(0, 18))), ch, fill=0, font=font)
            canvas = canvas.rotate(float(rng.normal(0, 4)), fillcolor=255)
            imgs.append(preprocess(ImageOps.crop(canvas, 2)))
            labels.append(ch)
    return np.stack(imgs)[..., None], np.array(labels)

if os.path.exists(CSV_PATH):
    glyphs, glyph_labels = load_real_dataset(); DATA_KIND = "real"
else:
    print("!! Handwriting dataset not found - using SYNTHETIC font glyphs (smoke test only).")
    glyphs, glyph_labels = load_synthetic_dataset(); DATA_KIND = "synthetic"

ch2cls = {c: i for i, c in enumerate(CHARS)}
glyph_cls = np.array([ch2cls[c] for c in glyph_labels], dtype=np.int32)
N_CLASSES = len(CHARS)
print(DATA_KIND, "| images:", glyphs.shape, "| classes:", len(set(glyph_cls)))

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, i in zip(axes.ravel(), np.random.choice(len(glyphs), 16, replace=False)):
    ax.imshow(1 - glyphs[i, ..., 0], cmap="gray", vmin=0, vmax=1); ax.set_title(glyph_labels[i]); ax.axis("off")
plt.suptitle("Training samples"); plt.show()

### 2.2 Conditional VAE
* **Encoder:** image + character label → latent style vector `z` (16-D).
* **Decoder:** `z` + character label → glyph image.

After training we throw the encoder away: drawing `z ~ N(0, I)` and asking the decoder for the letter "a" gives a *new* handwritten "a"
that never appeared in the dataset. Different `z` = different pen pressure, slant, wobble. Training uses random shifts, rotations
and zooms because 55 images per class is very little data.

In [ ]:
def build_encoder():
    img = layers.Input((IMG, IMG, 1)); lab = layers.Input((), dtype="int32")
    x = img
    for f in (32, 64, 128):
        x = layers.Conv2D(f, 4, strides=2, padding="same", activation="relu")(x)      # 64 -> 8
    x = layers.Concatenate()([layers.Flatten()(x), layers.Embedding(N_CLASSES, 32)(lab)])
    h = layers.Dense(256, activation="relu")(x)
    return Model([img, lab], [layers.Dense(LATENT)(h), layers.Dense(LATENT)(h)], name="encoder")

def build_decoder():
    z = layers.Input((LATENT,)); lab = layers.Input((), dtype="int32")
    h = layers.Concatenate()([z, layers.Embedding(N_CLASSES, 32)(lab)])
    h = layers.Reshape((8, 8, 128))(layers.Dense(8 * 8 * 128, activation="relu")(h))
    for f in (128, 64, 32):
        h = layers.Conv2DTranspose(f, 4, strides=2, padding="same", activation="relu")(h)  # 8 -> 64
    return Model([z, lab], layers.Conv2D(1, 3, padding="same", activation="sigmoid")(h), name="decoder")

encoder, decoder = build_encoder(), build_decoder()
augment = tf.keras.Sequential([
    layers.RandomTranslation(0.08, 0.08, fill_mode="constant", fill_value=0.0),
    layers.RandomRotation(0.04, fill_mode="constant", fill_value=0.0),
    layers.RandomZoom(0.1, fill_mode="constant", fill_value=0.0)])
opt = tf.keras.optimizers.Adam(1e-3)

@tf.function
def train_step(x, y, beta):
    x = augment(x, training=True)
    with tf.GradientTape() as tape:
        mu, logvar = encoder([x, y], training=True)
        z = mu + tf.exp(0.5 * logvar) * tf.random.normal(tf.shape(mu))
        xh = decoder([z, y], training=True)
        eps = 1e-6
        rec = -tf.reduce_sum(x * tf.math.log(xh + eps) + (1 - x) * tf.math.log(1 - xh + eps), axis=[1, 2, 3])
        kl = -0.5 * tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=1)
        loss = tf.reduce_mean(rec + beta * kl)
    vars_ = encoder.trainable_variables + decoder.trainable_variables
    opt.apply_gradients(zip(tape.gradient(loss, vars_), vars_))
    return tf.reduce_mean(rec), tf.reduce_mean(kl)

In [ ]:
ds = tf.data.Dataset.from_tensor_slices((glyphs, glyph_cls)).shuffle(len(glyphs)).batch(VAE_BATCH, drop_remainder=True)
log = []
for epoch in range(VAE_EPOCHS):
    beta = tf.constant(KL_BETA * min(1.0, (epoch + 1) / max(1, VAE_EPOCHS // 5)), tf.float32)   # KL warm-up avoids posterior collapse
    r, k = [], []
    for xb, yb in ds:
        a, b = train_step(xb, yb, beta); r.append(a.numpy()); k.append(b.numpy())
    log.append((np.mean(r), np.mean(k)))
    if epoch % max(1, VAE_EPOCHS // 15) == 0 or epoch == VAE_EPOCHS - 1:
        print(f"epoch {epoch+1:3d}/{VAE_EPOCHS}  reconstruction {log[-1][0]:7.1f}  KL {log[-1][1]:5.1f}")

log = np.array(log)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(log[:, 0]); ax[0].set_title("Reconstruction loss"); ax[1].plot(log[:, 1]); ax[1].set_title("KL (should stay > 0)")
plt.tight_layout(); plt.show()

### 2.3 Do the glyphs look right, and do they vary?
Each row is one character; each column is a different style vector. If every column in a row looks identical the VAE has collapsed
to a class average (the original notebook's problem) - check that KL above is not ~0.

In [ ]:
def sample_glyphs(chars, z=None, temp=1.0):
    cls = np.array([ch2cls[c] for c in chars], dtype=np.int32)
    if z is None:
        z = np.random.randn(len(chars), LATENT).astype(np.float32) * temp
    return decoder([z, cls], training=False).numpy()[..., 0]

demo = list("aAgQ7e")
fig, axes = plt.subplots(len(demo), 8, figsize=(10, 1.4 * len(demo)))
for r, ch in enumerate(demo):
    for c, g in enumerate(sample_glyphs([ch] * 8)):
        axes[r, c].imshow(1 - g, cmap="gray", vmin=0, vmax=1); axes[r, c].axis("off")
    axes[r, 0].set_title(ch, loc="left", fontsize=9)
plt.suptitle("Generated glyphs: rows = character, columns = different style samples"); plt.show()

# Part 3 - Putting it together: write new text in handwriting

`render_text` draws a line (or several) of text. One **style vector per page** is shared by all characters so the writing looks like
it came from a single hand; a little per-character jitter and baseline wobble keep it from looking printed. Each glyph is cropped
to its ink horizontally so letters sit close together.

In [ ]:
def crop_ink(g, pad=2, thr=0.15):
    cols = np.where(g.max(axis=0) > thr)[0]
    return g[:, max(cols[0] - pad, 0): cols[-1] + pad + 1] if len(cols) else g[:, : IMG // 3]

def render_text(text, style_jitter=0.25, style_scale=0.8, max_chars=28, space_px=22, seed=None, wobble=3):
    if seed is not None: np.random.seed(seed)
    text = "".join(c for c in text if c in c2i)                    # keep only supported characters
    lines, cur = [], ""
    for w in text.split():                                         # greedy word wrap
        if len(cur) + len(w) + 1 > max_chars and cur: lines.append(cur); cur = w
        else: cur = (cur + " " + w).strip()
    if cur: lines.append(cur)

    style = np.random.randn(LATENT).astype(np.float32) * style_scale
    rows = []
    for line in lines:
        pieces = []
        for ch in line:
            if ch == " ":
                pieces.append(np.zeros((IMG + 2 * wobble, space_px), np.float32)); continue
            z = (style + style_jitter * np.random.randn(LATENT)).astype(np.float32)[None]
            g = crop_ink(sample_glyphs([ch], z)[0])
            tile = np.zeros((IMG + 2 * wobble, g.shape[1]), np.float32)
            off = np.random.randint(0, 2 * wobble + 1)
            tile[off: off + IMG] = g
            pieces.append(tile)
        rows.append(np.concatenate(pieces, axis=1))
    W = max(r.shape[1] for r in rows)
    page = np.concatenate([np.pad(r, ((0, 0), (8, W - r.shape[1] + 8))) for r in rows], axis=0)
    return 1.0 - np.clip(page, 0, 1)                               # white paper, dark ink

def show(img, title=None, scale=0.045):
    plt.figure(figsize=(img.shape[1] * scale, img.shape[0] * scale))
    plt.imshow(img, cmap="gray", vmin=0, vmax=1); plt.axis("off")
    if title: plt.title(title, fontsize=9)
    plt.show()

show(render_text("Hello World 2026", seed=1), "Typed text, rendered by the glyph model")

### Final demo: the RNN writes, the glyph model draws

In [ ]:
generated = generate_text("The ", n_chars=110, temperature=0.7)
print("RNN wrote:", generated)
page = render_text(generated, seed=7)
show(page, "RNN-generated text in generated handwriting")
Image.fromarray((page * 255).astype(np.uint8)).save("generated_handwriting.png")

### Same text, different "writers"
A new style vector is a new hand.

In [ ]:
for s in (11, 12, 13):
    show(render_text("Quick brown fox 123", seed=s), f"writer {s - 10}")

In [ ]:
rnn.save("char_rnn.keras"); decoder.save("glyph_decoder.keras")
print("saved char_rnn.keras, glyph_decoder.keras, generated_handwriting.png")

## Notes and limitations
* **Text RNN quality depends on the corpus.** A few MB of text and 25+ epochs gives plausible words; a tiny corpus gives memorisation.
  Try `temperature` 0.5-1.0.
* **Glyphs are independent images**, not pen trajectories, so there is no ligature/joined-up cursive and lowercase/uppercase size
  differences are lost (each source photo is scaled to fill the frame). Modelling real pen strokes (Graves-style mixture-density RNN) needs
  *online* stroke data such as IAM-OnDB, which this dataset does not contain.
* **Glyphs are slightly soft** - the usual VAE trade-off with only ~55 examples per class. Lower `KL_BETA` for sharper strokes (less
  variety), raise it for more variety (blurrier), or swap the decoder for a conditional GAN if you need crisper output.
* Characters outside a-z, A-Z, 0-9 and space are dropped, since the dataset has no punctuation.